In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets scikit-learn
!pip install -q -U --force-reinstall pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.4 MB/s eta 0:00:00


Drive Mount + Dataset Load

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import json

df = pd.read_csv("/content/drive/MyDrive/Stanza_AI_Intent_Classification_Final_700.csv")

def build_prompt(query):
    return f"""You are an intent classifier for a hotel AI system called Stanza AI. Given a guest or staff query, identify the responsible agent and the specific intent.

Query: {query}
Answer (JSON only):"""

def build_target(agent_label, intent_name):
    return json.dumps({"agent_label": agent_label, "intent_name": intent_name})

df["prompt"] = df["query"].apply(build_prompt)
df["target"] = df.apply(lambda r: build_target(r["agent_label"], r["intent_name"]), axis=1)
df["full_text"] = df["prompt"] + " " + df["target"]

from datasets import Dataset
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df[["full_text"]])
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

Mounted at /content/drive
Train: 560 Val: 70 Test: 70


Mistral Model Load (Base)

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Mistral base model loaded.")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Mistral base model loaded.


LoRA Setup

In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 7,261,655,040 || trainable%: 0.1877


Training

In [5]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./mistral-stanza-finetune",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    max_length=512,
    dataset_text_field="full_text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_dataset,
    eval_dataset=Dataset.from_pandas(val_df[["full_text"]]),
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/560 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/70 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.289531,0.284188,0.262953,44726.000000,0.931590
2,0.205679,0.250468,0.218536,89452.000000,0.937045
3,0.156007,0.257450,0.173588,134178.000000,0.938210


TrainOutput(global_step=210, training_loss=0.3017125163759504, metrics={'train_runtime': 3760.4096, 'train_samples_per_second': 0.447, 'train_steps_per_second': 0.056, 'total_flos': 5875944849260544.0, 'train_loss': 0.3017125163759504, 'epoch': 3.0})

Drive Backup

In [6]:
import shutil, os
print(os.listdir("./mistral-stanza-finetune"))

shutil.copytree(
    "./mistral-stanza-finetune/checkpoint-210",
    "/content/drive/MyDrive/stanza_ai_mistral_checkpoint",
    dirs_exist_ok=True
)
print("Mistral backed up to Drive!")

['checkpoint-210', 'checkpoint-140', 'checkpoint-70', 'README.md']
Mistral backed up to Drive!


Evaluation

In [7]:
import re, json as jsonlib
from tqdm import tqdm

model.eval()

def generate_answer(prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated.strip()

def extract_json(text):
    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if match:
        try:
            return jsonlib.loads(match.group())
        except:
            return None
    return None

results = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    raw_output = generate_answer(row["prompt"])
    parsed = extract_json(raw_output)
    results.append({
        "true_agent": row["agent_label"], "true_intent": row["intent_name"],
        "pred_agent": parsed.get("agent_label") if parsed else None,
        "pred_intent": parsed.get("intent_name") if parsed else None,
    })

mistral_results_df = pd.DataFrame(results)
mistral_results_df.to_csv("/content/drive/MyDrive/mistral_evaluation_results.csv", index=False)

100%|██████████| 70/70 [08:22<00:00,  7.18s/it]


Accuracy

In [9]:
mistral_results_df["fully_correct"] = (mistral_results_df["pred_agent"] == mistral_results_df["true_agent"]) & (mistral_results_df["pred_intent"] == mistral_results_df["true_intent"])
overall = mistral_results_df["fully_correct"].mean() * 100
agent_acc = (mistral_results_df["pred_agent"] == mistral_results_df["true_agent"]).mean() * 100
intent_acc = (mistral_results_df["pred_intent"] == mistral_results_df["true_intent"]).mean() * 100

print(f"MISTRAL — Overall: {overall:.2f}% | Agent: {agent_acc:.2f}% | Intent: {intent_acc:.2f}%")

MISTRAL — Overall: 84.29% | Agent: 91.43% | Intent: 85.71%
